# 16S Analyses of the Longitudinal Acne Study
## Random Forest analysis

Date created: 12/21/2024

Notebook authors: Yang Chen

Data analysis by: Tyler Myers, Britta De Pessemier, and Yang Chen

This notebook plots the following:

- Random forest analysis showing the AUCs for separating the three groups for each primer pair and the driving taxa.

In [14]:
# Import Python packages
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.utils import resample
import numpy as np
import pandas as pd
import qiime2
import matplotlib.pyplot as plt

In [15]:
def qza_to_dataframe(qza_path, metadata_path, column):
    """
    Converts a QIIME 2 .qza table to a pandas DataFrame and appends a column from metadata.

    Parameters:
    qza_path (str): Path to the QIIME 2 .qza file.
    metadata_path (str): Path to the metadata CSV file.
    column (str): Name of the column in the metadata to be added as the label.

    Returns:
    pd.DataFrame: A DataFrame with features as rows, samples as columns, and the category label as the last column.
    """
    import qiime2
    import pandas as pd

    # Load the QIIME 2 Artifact
    table_artifact = qiime2.Artifact.load(qza_path)

    # Export the BIOM table to a pandas DataFrame
    biom_table = table_artifact.view(pd.DataFrame)

    # Load metadata
    metadata = pd.read_csv(metadata_path, index_col=0)  # Assuming sample IDs are in the index

    # Ensure sample IDs in BIOM table match those in the metadata
    # if not biom_table.index.equals(metadata.index):
    #     raise ValueError("Sample IDs in the BIOM table and metadata do not match.")

    # Add the category label column from metadata
    biom_table[column] = metadata[column]

    # Remove rows with NaN values in the specified column
    biom_table = biom_table.dropna(subset=[column])

    return biom_table


# Load data for V1-V3 and V4
v1_v3_data = qza_to_dataframe('../Data/16S/Tables/16S_V1-V3_rarefied_uncollapsed_table.qza', '../Metadata/57610_57610_analysis_mapping_category_label.csv', 'category_label')
v4_data = qza_to_dataframe('../Data/16S/Tables/16S_V4_rarefied_uncollapsed_table.qza', '../Metadata/57610_57610_analysis_mapping_category_label.csv', 'category_label')

v1_v3_data


,GATGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCACAGCTTGCTGTGGTACTCGAGTGGCGAACGGGTGAGTAACACGTGGGTGATCTGCCCTGTACTTCGGGATAAGCTTGGGAAACTGGGTCTAATACCGGAT,AGCGAACGCTGGCGGCAGGCTTAACACATGCAAGTCGAGCGCCGTAGCAATACGGAGCGGCAGACGGGTGAGTAACACGTGGGAACGTACCTTTTGGTTCGGAACAACACAGGGAAACTTGTGCTAATACCGGATAAGCCCTTACGGGGA,ATTGAACGCTGGCGGCATGCCTTACACATGCAAGTCGAACGGTAACAGGTCTTCGGATGCTGACGAGTGGCGAACGGGTGAGTAATACATCGGAACGTGCCCGATCGTGGGGGATAACGGAGCGAAAGCTTTGCTAATACCGCATACGAT,GATGAACGCTAGCGATAGGCTTAACACATGCAAGTCGAGGGGCAGCATGGTCTTAGCTTGCTAAGACTGATGGCGACCGGCGCACGGGTGCGTAACGCGTATGCAACTTGCCTCACAGAGGGGGATAACCCGTCGAAAGACGGACTAATA,GACGAACGCTGGCGGCGCGCCTAACACATGCAAGTCGAACGGAGCTTAGAGAGCTTGCTTTTTAAGCTTAGTGGCGAACGGGTGAGTAACGCGTGAGTAACCTGCCCTAGAGTGGGGGACAACAGTTGGAAACGACTGCTAATACCGCAT,ATTGAACGCTGGCGGCAGGCTTAACACATGCAAGTCGAACGATGATTATCTAGCTTGCTAGATATGATTAGTGGCGGACGGGTGAGTAACATTTAGGAATCTGCCTAGTAGTGGGGGATAGCTCGGGGAAACTCGAATTAATACCGCATA,ATTGAACGCTGGCGGCATGCTTTACACATGCAAGTCGGACGGCAGCACAGAGAAGCTTGCTTCTTGGGTGGCGAGTGGCGAACGGGTGAGTAATATATCGGAACGTACCGAGTAATGGGGGATAACTAATCGAAAGATTAGCTAATACCG,AGTGAACGCTGGCGGCAGGCCTAACACATGCAAGTCGAACGGCAGCACAGTAAGAGCTTGCTCTTATGGGTGGCGAGTGGCGGACGGGTGAGGAATACATCGGAATCTACTCTTTCGTGGGGGATAACGTAGGGAAACTTACGCTAATAC,ATTGAACGCTGGCGGCAGGCCTAACACATGCAAGTCGAGCGGATGAGTGGAGCTTGCTCCATGATTCAGCGGCGGACGGGTGAGTAATGCCTAGGAATCTGCCTGGTAGTGGGGGACAACGTTTCGAAAGGAACGCTAATACCGCATACG,GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTGCAGGGTACTCGAGTGGCGAACGGGTGAGTAACACGTGGGTGATCTGCCTCGTACTTCGGGATAAGCCTGGGAAACTGGGTCTAATACCGGATAG,...,GATGAACGCTGGCGGCGTGCCTAATACATGCAAGTCGAGCGAACAGACGAGGAGCTTGCTCCTCTGACGTTAGCGGCGGACGGGTGAGTAACACGTGGATAACCTACCTATAAGACTGGGATAACTTCGGGAAACCGGAGCTAATACCGG,GATGAACGCTGGCGGCGTGCCTAATACATGCAAGTCGAGCGAACAGACGAGGAGCTTGCTCCTTTGACGTTAGCGGCGGACGGGTGAGTAACACGTGGATAACCTACCTATAAGACTGGGATAACTTCGGGAAACCGGAGCTAATACCGG,GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGTACGGTAAGGCCCTTTCGGGGGTACACGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCACAACTTTGGGATAACGCTAGGAAACTGGTGCTAATACTGGATATGT,ATTGAACGCTGGCGGCATGCTTTACACATGCAAGTCGAACGGCAGCGGGGAGGAGCTTGCTTCTCTGCCGGCGAGTGGCGAACGGGTGAGTAATACATCGGAACGTGTCGAGAAGAGGGGGATAACCACCCGAAAGGGTGGCTAATACCG,GATGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGCTGAAGCCTGGTGCTTGCACTGGGTGGATGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTCGGGATAAGCCCGGGAAACTGGGTCTAATACTGG,GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGCTGAAGCTTGGTGCTTGCACTGGGTGGATGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTTACTTCGGGATAAGCTTGGGAAACTGGGTCTAATACCGG,GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAGCGGAAAGGCCCTTCGGGGTACTCGAGCGGCGAACGGGTGAGTAACACGTGAGTAATCTGCCCCGTACTTCGGGATAGCCACCGGAAACGGTGATTAATACCGGATACGACG,GACGAACGCTGGCGGCGTGCCTAATACATGCAAGTAGAACGCTGAAGCTTGGTGCTTGCACCGAGCGGATGAGTTGCGAACGGGTGAGTAACGCGTAGGTAACCTGCCTGGTAGCGGGGGATAACTATTGGAAACGATAGCTAATACCGC,GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGTAAGGCCCCAGCTTGCTGGGGTACACGAGTGGCGAACGGGTGAGTAACACGTGGGTGACCTGCCCCGCACTTCGGGATAAGCCTGGGAAACTGGGTCTAATACCGGAT,category_label
173980.14901.LAMI.RD001.D0.C1,11.0,0.0,0.0,100.0,0.0,1.0,47.0,0.0,0.0,0.0,...,83.0,0.0,0.0,0.0,5.0,0.0,0.0,39.0,24.0,Healthy
173980.14901.LAMI.RD001.D0.C2,7.0,0.0,0.0,82.0,0.0,2.0,89.0,0.0,0.0,0.0,...,155.0,0.0,27.0,1.0,0.0,3.0,0.0,49.0,84.0,Healthy
173980.14901.LAMI.RD001.D14.C1,0.0,0.0,0.0,12.0,0.0,2.0,13.0,0.0,0.0,0.0,...,81.0,0.0,44.0,0.0,3.0,74.0,0.0,25.0,4.0,Healthy
173980.14901.LAMI.RD001.D14.C2,3.0,0.0,0.0,155.0,0.0,0.0,31.0,0.0,0.0,0.0,...,137.0,0.0,20.0,0.0,6.0,36.0,0.0,48.0,10.0,Healthy
173980.14901.LAMI.RD001.D28.C1,0.0,0.0,0.0,36.0,0.0,0.0,7.0,0.0,0.0,0.0,...,115.0,0.0,11.0,0.0,4.0,6.0,0.0,23.0,39.0,Healthy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173980.14901.LAMI.RD319.D14.C1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,8.0,0.0,11.0,0.0,0.0,0.0,0.0,0.0,0.0,Acne Lesional
173980.14901.LAMI.RD319.D21.C3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,8.0,0.0,13.0,0.0,0.0,0.0,0.0,0.0,0.0,Acne Non-lesional
173980.14901.LAMI.RD319.D14.C2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,8.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,Acne Lesional
173980.14901.L

In [16]:
def random_forest_auc_binary(features_df, n_estimators_list, label_column, categories):
    """
    Perform Random Forest classification and calculate AUC for two categories.

    Parameters:
    features_df (pd.DataFrame): DataFrame with features and a label column.
    n_estimators_list (list): List of numbers of trees for Random Forest.
    label_column (str): Name of the column containing class labels.
    categories (list): Two categories to filter and analyze.

    Returns:
    dict: AUC scores for each number of trees.
    """

    # Filter DataFrame for the specified categories
    filtered_df = features_df[features_df[label_column].isin(categories)]

    # Extract features and labels
    labels = filtered_df[label_column]
    features = filtered_df.drop(columns=[label_column])

    # Map the two categories to binary labels (e.g., 0 and 1)
    labels_binary = labels.map({categories[0]: 0, categories[1]: 1})

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        features, labels_binary, stratify=labels_binary, test_size=0.2, random_state=42
    )

    auc_scores = {}
    for n_trees in n_estimators_list:
        # Train Random Forest
        rf = RandomForestClassifier(n_estimators=n_trees, random_state=42)
        rf.fit(X_train, y_train)

        # Predict probabilities
        y_prob = rf.predict_proba(X_test)[:, 1]  # Get probabilities for the positive class

        # Calculate AUC
        auc = roc_auc_score(y_test, y_prob)
        auc_scores[n_trees] = auc

    return auc_scores


In [17]:
# Run Random Forest Analysis between Acne Lesional and Healthy
categories = ["Acne Lesional", "Healthy"]
n_estimators_list = [100, 500, 1000]
label_column = "category_label"

# Perform analysis for V1-V3
v1_v3_auc = random_forest_auc_binary(v1_v3_data, n_estimators_list, label_column, categories)
print("V1-V3 AUC scores (Acne Lesional vs. Healthy):", v1_v3_auc)

# Perform analysis for V4
v4_auc = random_forest_auc_binary(v4_data, n_estimators_list, label_column, categories)
print("V4 AUC scores (Acne Lesional vs. Healthy):", v4_auc)


V1-V3 AUC scores (Acne Lesional vs. Healthy): {100: 0.9846743295019157, 500: 0.9885057471264368, 1000: 0.996168582375479}
V4 AUC scores (Acne Lesional vs. Healthy): {100: 0.8500000000000001, 500: 0.8466666666666667, 1000: 0.8566666666666667}


In [18]:
# Run Random Forest Analysis between Acne Non-lesional and Healthy
categories = ["Acne Non-lesional", "Healthy"]
n_estimators_list = [100, 500, 1000]
label_column = "category_label"

# Perform analysis for V1-V3
v1_v3_auc = random_forest_auc_binary(v1_v3_data, n_estimators_list, label_column, categories)
print("V1-V3 AUC scores (Acne Non-lesional vs. Healthy):", v1_v3_auc)

# Perform analysis for V4
v4_auc = random_forest_auc_binary(v4_data, n_estimators_list, label_column, categories)
print("V4 AUC scores (Acne Non-lesional vs. Healthy):", v4_auc)


V1-V3 AUC scores (Acne Non-lesional vs. Healthy): {100: 0.9611111111111111, 500: 0.9555555555555555, 1000: 0.9444444444444444}
V4 AUC scores (Acne Non-lesional vs. Healthy): {100: 0.851851851851852, 500: 0.8703703703703703, 1000: 0.8703703703703703}


In [19]:
# Run Random Forest Analysis between Acne Lesional and Acne Non-lesional
categories = ["Acne Lesional", "Acne Non-lesional"]
n_estimators_list = [100, 500, 1000]
label_column = "category_label"

# Perform analysis for V1-V3
v1_v3_auc = random_forest_auc_binary(v1_v3_data, n_estimators_list, label_column, categories)
print("V1-V3 AUC scores (Acne Lesional vs. Acne Non-lesional):", v1_v3_auc)

# Perform analysis for V4
v4_auc = random_forest_auc_binary(v4_data, n_estimators_list, label_column, categories)
print("V4 AUC scores (Acne Lesional vs. Acne Non-lesional):", v4_auc)


V1-V3 AUC scores (Acne Lesional vs. Acne Non-lesional): {100: 0.6551724137931035, 500: 0.6819923371647509, 1000: 0.6781609195402298}
V4 AUC scores (Acne Lesional vs. Acne Non-lesional): {100: 0.49444444444444435, 500: 0.4851851851851852, 1000: 0.4666666666666667}


## ROC plot

In [29]:
def plot_roc_curves_with_ci(features_df, label_column, categories, n_estimators, primer_set_label, color, linestyle='-'):
    """
    Generate ROC curves with a fixed confidence interval (±0.05) for a specific primer set and category comparison.

    Parameters:
    features_df (pd.DataFrame): DataFrame containing features and labels.
    label_column (str): Column name containing the labels.
    categories (list): Two categories to filter and compare.
    n_estimators (int): Number of trees for Random Forest.
    primer_set_label (str): Label for the primer set (e.g., "V1-V3").
    color (str): Color for the ROC curve and confidence interval.
    linestyle (str): Line style for the ROC curve (default is solid line '-').
    """
    # Filter DataFrame for the specified categories
    filtered_df = features_df[features_df[label_column].isin(categories)]
    labels = filtered_df[label_column].map({categories[0]: 0, categories[1]: 1})
    features = filtered_df.drop(columns=[label_column])

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        features, labels, stratify=labels, test_size=0.2, random_state=42
    )

    # Train Random Forest
    rf = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
    rf.fit(X_train, y_train)

    # Predict probabilities
    y_prob = rf.predict_proba(X_test)[:, 1]

    # Calculate ROC curve and AUC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    # Define fixed confidence interval
    ci_margin = 0.05
    lower_tpr = np.maximum(tpr - ci_margin, 0)
    upper_tpr = np.minimum(tpr + ci_margin, 1)

    # Plot the ROC curve with fixed confidence intervals
    plt.plot(fpr, tpr, label=f"{primer_set_label} (AUC = {roc_auc:.2f} ± 0.05)", color=color, linestyle=linestyle, linewidth=2.5)
    plt.fill_between(fpr, lower_tpr, upper_tpr, color=color, alpha=0.2)


# Categories for comparisons
categories_lesional_vs_healthy = ["Acne Lesional", "Healthy"]
categories_lesional_vs_non_lesional = ["Acne Lesional", "Acne Non-lesional"]
categories_non_lesional_vs_healthy = ["Acne Non-lesional", "Healthy"]

# Number of trees for Random Forest
n_estimators = 1000

# Create ROC plot
plt.figure(figsize=(9, 8))

# V4 data (dotted lines)
plot_roc_curves_with_ci(v4_data, "category_label", categories_lesional_vs_non_lesional, n_estimators, "V4 AL vs ANL", "#f16c52", linestyle="--")
plot_roc_curves_with_ci(v4_data, "category_label", categories_non_lesional_vs_healthy, n_estimators, "V4 H vs ANL", "#5cbccb", linestyle="--")
plot_roc_curves_with_ci(v4_data, "category_label", categories_lesional_vs_healthy, n_estimators, "V4 H vs AL)", "#3333B3", linestyle="--")

# V1-V3 data (solid lines)
plot_roc_curves_with_ci(v1_v3_data, "category_label", categories_lesional_vs_non_lesional, n_estimators, "V1-V3 AL vs ANL", "#f16c52")
plot_roc_curves_with_ci(v1_v3_data, "category_label", categories_non_lesional_vs_healthy, n_estimators, "V1-V3 H vs ANL", "#5cbccb")
plot_roc_curves_with_ci(v1_v3_data, "category_label", categories_lesional_vs_healthy, n_estimators, "V1-V3 H vs AL", "#3333B3")

# Customize plot
plt.plot([0, 1], [0, 1], color="black", linestyle="--", label="Random Classifier")
plt.xlabel("False Positive Rate", fontsize=18)
plt.ylabel("True Positive Rate", fontsize=18)
plt.title("V1-V3 vs V4 Random Forest ROC Curves", fontsize=20)

# Define the desired legend order
desired_order = [
    "V1-V3 H vs AL (AUC = 1.00 ± 0.05)",
    "V4 H vs AL) (AUC = 0.86 ± 0.05)",
    "V1-V3 H vs ANL (AUC = 0.94 ± 0.05)",
    "V4 H vs ANL (AUC = 0.87 ± 0.05)",
    "V1-V3 AL vs ANL (AUC = 0.68 ± 0.05)",
    "V4 AL vs ANL (AUC = 0.47 ± 0.05)"
]

# Get current handles and labels from the legend
handles, labels = plt.gca().get_legend_handles_labels()

# Reorder handles and labels based on the desired order
sorted_handles = []
sorted_labels = []
for label in desired_order:
    if label in labels:
        index = labels.index(label)
        sorted_handles.append(handles[index])
        sorted_labels.append(labels[index])

# Apply the reordered legend
plt.legend(sorted_handles, sorted_labels, loc="lower right", fontsize=12)

# Finalize and show the plot
plt.grid()
plt.tight_layout()
plt.savefig('../Figures/16S_Figures/primer_comparison/roc_curves_with_ci.png', dpi=600)
